# 04 - Análisis Descriptivo
## Hotel Dann Monasterio - cálculo de KPIs y validación de hipótesis

## Objetivo del notebook
Cierre analítico del proyecto. A diferencia del EDA (notebook 03) que explora libremente los datos, aquí formalizamos los hallazgos en:

1. **10 KPIs** definidos en el Capítulo 4 del anteproyecto.
2. **Validación de las 8 hipótesis** (H1-H8) formuladas en el Capítulo 5.
3. **Análisis comparativo** pre-pandemia / pandemia / recuperación / post-pandemia.
4. **Resumen ejecutivo** con conclusiones accionables.

## Contexto CRISP-DM
Este notebook corresponde a la fase **Evaluación**. Sus resultados son el insumo directo para los dashboards y para el informe final.

## Importar librerías

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

plt.rcParams["figure.figsize"] = (12, 6)
sns.set_style("whitegrid")
sns.set_palette("Set2")
pd.set_option("display.max_columns", 80)
pd.options.display.float_format = "{:,.2f}".format

## Cargar dataset limpio

In [ ]:
ruta = Path("../data/processed/reservas_clean.parquet")
df = pd.read_parquet(ruta) if ruta.exists() else pd.read_csv(ruta.with_suffix(".csv"))
print(f"Shape: {df.shape}")
df.head(3)

## Carpeta para figuras y reporte

Los resultados textuales se acumulan en `reports/evaluacion_hipotesis.md`.

In [ ]:
FIG_DIR = Path("../reports/figures")
REPORT_PATH = Path("../reports/evaluacion_hipotesis.md")
FIG_DIR.mkdir(parents=True, exist_ok=True)
REPORT_PATH.parent.mkdir(parents=True, exist_ok=True)

report_lines = ["# Reporte de evaluación de hipótesis - Hotel Dann Monasterio", ""]

## Parámetros del hotel

Para calcular ocupación y RevPAR necesitamos el inventario del hotel.

In [ ]:
TOTAL_HABITACIONES = 47   # según el dataset (df['nrohab_hab'].nunique())
print(f"Número de habitaciones únicas en el dataset: {df['nrohab_hab'].nunique()}")
print(f"Usaremos TOTAL_HABITACIONES = {TOTAL_HABITACIONES}")

# 1. Cálculo de los 10 KPIs

## KPI-01 - Ingresos totales

Suma de `totalconsumosplan` + `totalconsumosadicional` para todo el periodo y por año.

In [ ]:
kpi01_total   = df["ingreso_total"].sum()
kpi01_por_año = df.groupby("anio")["ingreso_total"].sum()

print(f"KPI-01 - Ingresos totales en el periodo: ${kpi01_total:,.0f} COP")
print("\nIngresos por año:")
print(kpi01_por_año.apply(lambda x: f"${x:,.0f}"))

## KPI-02 - ADR (Average Daily Rate)

ADR = Σ totalconsumosplan / Σ noches ocupadas

In [ ]:
noches_totales = df["duracion_estancia"].clip(lower=0).sum()
kpi02_adr = df["totalconsumosplan"].sum() / noches_totales if noches_totales else 0
print(f"KPI-02 - ADR (tarifa media diaria): ${kpi02_adr:,.0f} COP / noche")
print(f"           Noches ocupadas en el periodo: {noches_totales:,.0f}")

## KPI-03 - Tasa de ocupación

Ocupación = Σ noches ocupadas / (TOTAL_HABITACIONES × días del periodo)

In [ ]:
periodo_dias = (df["fllega_aco"].max() - df["fllega_aco"].min()).days + 1
capacidad = TOTAL_HABITACIONES * periodo_dias
kpi03_ocupacion = noches_totales / capacidad * 100 if capacidad else 0
print(f"KPI-03 - Tasa de ocupación: {kpi03_ocupacion:.2f}%")
print(f"           Días del periodo: {periodo_dias:,}")
print(f"           Capacidad teórica (hab × días): {capacidad:,}")

## KPI-04 - RevPAR (Revenue Per Available Room)

RevPAR = ADR × tasa de ocupación / 100

In [ ]:
kpi04_revpar = kpi02_adr * kpi03_ocupacion / 100
print(f"KPI-04 - RevPAR: ${kpi04_revpar:,.0f} COP")

## KPI-05 - Duración promedio de estancia

In [ ]:
kpi05 = df[df["duracion_estancia"] >= 0]["duracion_estancia"].mean()
print(f"KPI-05 - Duración promedio de estancia: {kpi05:.2f} noches")

print("\nPor segmento:")
print(df[df["duracion_estancia"]>=0].groupby("codsegmento")["duracion_estancia"].mean().sort_values(ascending=False))

## KPI-06 - Lead time promedio

In [ ]:
kpi06 = df[df["lead_time"].between(0, 365)]["lead_time"].mean()
print(f"KPI-06 - Lead time promedio (0-365 días): {kpi06:.2f} días")

print("\nPor canal (top 8):")
top8 = df["nombre_age"].value_counts().head(8).index
lt_canal = df[df["nombre_age"].isin(top8) & df["lead_time"].between(0,365)].groupby("nombre_age")["lead_time"].mean().sort_values(ascending=False)
print(lt_canal)

## KPI-07 - Participación de ingresos por segmento

In [ ]:
kpi07 = df.groupby("codsegmento")["ingreso_total"].sum().sort_values(ascending=False)
kpi07_pct = (kpi07 / kpi07.sum() * 100).round(2)
kpi07_resumen = pd.DataFrame({"ingresos": kpi07, "% ingresos": kpi07_pct})
kpi07_resumen

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))
ax.pie(kpi07.values, labels=kpi07.index, autopct="%1.1f%%", startangle=90)
ax.set_title("KPI-07 - Composición de ingresos por segmento")
plt.tight_layout()
plt.savefig(FIG_DIR / "kpi07_segmento.png", dpi=120)
plt.show()

## KPI-08 - Participación de ingresos por canal

In [ ]:
kpi08 = df.groupby("nombre_age")["ingreso_total"].sum().sort_values(ascending=False).head(10)
kpi08_pct = (kpi08 / df["ingreso_total"].sum() * 100).round(2)
kpi08_resumen = pd.DataFrame({"ingresos": kpi08, "% ingresos del total": kpi08_pct})
kpi08_resumen

## KPI-09 - Ticket promedio por reserva

In [ ]:
reservas_unicas = df["numvoucher"].nunique()
kpi09 = df["ingreso_total"].sum() / reservas_unicas if reservas_unicas else 0
print(f"KPI-09 - Ticket promedio por reserva: ${kpi09:,.0f} COP")
print(f"           Reservas únicas (numvoucher): {reservas_unicas:,}")

## KPI-10 - % consumos adicionales sobre ingreso total

In [ ]:
kpi10 = df["totalconsumosadicional"].sum() / df["ingreso_total"].sum() * 100
print(f"KPI-10 - Participación de consumos adicionales: {kpi10:.2f}%")

## Cuadro resumen de los 10 KPIs

Consolidamos los indicadores en una tabla para que sea consumible por el dashboard.

In [ ]:
resumen_kpis = pd.DataFrame({
    "KPI": [
        "KPI-01 Ingresos totales",
        "KPI-02 ADR",
        "KPI-03 Tasa de ocupación (%)",
        "KPI-04 RevPAR",
        "KPI-05 Duración promedio (noches)",
        "KPI-06 Lead time promedio (días)",
        "KPI-07 # segmentos > 10% ingresos",
        "KPI-08 # canales > 10% ingresos",
        "KPI-09 Ticket promedio por reserva",
        "KPI-10 % consumos adicionales",
    ],
    "valor": [
        kpi01_total,
        kpi02_adr,
        kpi03_ocupacion,
        kpi04_revpar,
        kpi05,
        kpi06,
        (kpi07_pct > 10).sum(),
        (kpi08_pct > 10).sum(),
        kpi09,
        kpi10,
    ]
})
resumen_kpis

In [ ]:
resumen_kpis.to_csv("../reports/kpis_resumen.csv", index=False)
print('Resumen guardado en ../reports/kpis_resumen.csv')

# 2. Validación de hipótesis

## H1 - El segmento corporativo (COR/EM) concentra más del 50% de los ingresos

Evidencia: % de ingresos por segmento.

In [ ]:
pct_cor = kpi07_pct.reindex(["COR", "EM"]).fillna(0).sum()
veredicto_h1 = "CONFIRMADA" if pct_cor > 50 else "REFUTADA"
linea = f"H1: ingresos COR+EM = {pct_cor:.2f}% -> {veredicto_h1}"
print(linea)
report_lines.append("## H1\n" + linea + "\n")

## H2 - Booking.com + Ventas Directas Recepción concentran > 70% de las reservas

Evidencia: % de reservas (numvoucher) por canal.

In [ ]:
reservas_canal = df.groupby("nombre_age")["numvoucher"].nunique().sort_values(ascending=False)
reservas_canal_pct = reservas_canal / reservas_canal.sum() * 100

claves = [c for c in reservas_canal_pct.index if ("BOOKING" in c.upper()) or ("RECEPCION" in c.upper())]
pct_h2 = reservas_canal_pct[claves].sum()
veredicto_h2 = "CONFIRMADA" if pct_h2 > 70 else "REFUTADA"
linea = f"H2: Booking+Recepción concentran {pct_h2:.2f}% de las reservas -> {veredicto_h2}"
print(linea)
print("\nCanales considerados:", claves)
report_lines.append("## H2\n" + linea + "\n")

## H3 - ADR significativamente mayor en temporada alta que en baja

Evidencia: ADR por temporada.

In [ ]:
adr_temp = df.dropna(subset=["nombretemporada"]).groupby("nombretemporada").apply(
    lambda g: g["totalconsumosplan"].sum() / max(g["duracion_estancia"].clip(lower=0).sum(), 1)
)
print("ADR por temporada:")
print(adr_temp)

if {"ALTA","BAJA"}.issubset(adr_temp.index):
    dif = adr_temp["ALTA"] - adr_temp["BAJA"]
    veredicto_h3 = "CONFIRMADA" if dif > 0 else "REFUTADA"
    linea = f"H3: ADR_alta=${adr_temp['ALTA']:,.0f} vs ADR_baja=${adr_temp['BAJA']:,.0f} (dif=${dif:,.0f}) -> {veredicto_h3}"
else:
    linea = "H3: no se puede evaluar (faltan etiquetas ALTA/BAJA en algunos registros)."
    veredicto_h3 = "INDETERMINADA"
print(linea)
report_lines.append("## H3\n" + linea + "\n")

## H4 - Duración promedio de estancia es menor en corporativos que en turistas

Evidencia: media de `duracion_estancia` por segmento.

In [ ]:
dur_seg = df[df["duracion_estancia"]>=0].groupby("codsegmento")["duracion_estancia"].mean()
print("Duración promedio por segmento:")
print(dur_seg.sort_values())

corp = dur_seg.reindex(["COR","EM"]).dropna().mean()
tur  = dur_seg.reindex(["T&T","PAR"]).dropna().mean()
veredicto_h4 = "CONFIRMADA" if corp < tur else "REFUTADA"
linea = f"H4: dur_corporativos={corp:.2f}n vs dur_turistas={tur:.2f}n -> {veredicto_h4}"
print(linea)
report_lines.append("## H4\n" + linea + "\n")

## H5 - Lead time mayor en agencias internacionales que en ventas directas

Evidencia: lead time promedio por canal.

In [ ]:
lt = df[df["lead_time"].between(0,365)].groupby("nombre_age")["lead_time"].mean()
print("Lead time promedio por canal (top 10 por reservas):")
print(lt.reindex(reservas_canal.head(10).index).dropna().sort_values(ascending=False))

ldir = lt.filter(regex="(?i)recepcion|directa").mean()
lint = lt.filter(regex="(?i)booking|expedia|aviatur|despegar|hotelbeds").mean()
if pd.notna(ldir) and pd.notna(lint):
    veredicto_h5 = "CONFIRMADA" if lint > ldir else "REFUTADA"
    linea = f"H5: lead_canal_internacional={lint:.1f}d vs lead_recepcion={ldir:.1f}d -> {veredicto_h5}"
else:
    veredicto_h5 = "INDETERMINADA"
    linea = "H5: no se encontraron suficientes canales para evaluar."
print(linea)
report_lines.append("## H5\n" + linea + "\n")

## H6 - El grupo demográfico 36-50 años es el mayoritario

Evidencia: distribución de `rango_edad`.

In [ ]:
rangos = df["rango_edad"].value_counts(normalize=True).mul(100).round(2)
print("Distribución por rango de edad (%):")
print(rangos)

veredicto_h6 = "CONFIRMADA" if rangos.idxmax() == "36-50" else "REFUTADA"
linea = f"H6: rango mayoritario = {rangos.idxmax()} ({rangos.max():.2f}%) -> {veredicto_h6}"
print(linea)
report_lines.append("## H6\n" + linea + "\n")

## H7 - Meses con mayor ocupación: enero, julio y diciembre

Evidencia: noches ocupadas por mes.

In [ ]:
noches_mes = df.groupby("mes")["duracion_estancia"].sum().sort_values(ascending=False)
print("Noches ocupadas por mes (orden descendente):")
print(noches_mes)

top3 = set(noches_mes.head(3).index)
esperados = {1, 7, 12}
veredicto_h7 = "CONFIRMADA" if top3 == esperados else ("PARCIAL" if top3 & esperados else "REFUTADA")
linea = f"H7: top3 meses = {sorted(top3)}, esperados = {sorted(esperados)} -> {veredicto_h7}"
print(linea)
report_lines.append("## H7\n" + linea + "\n")

## H8 - Planes con alimentación incluida generan mayor consumo adicional

Evidencia: promedio de `totalconsumosadicional` por valor de `alimen_pla` (S/N).

In [ ]:
cons_alim = df.groupby("alimen_pla")["totalconsumosadicional"].mean()
print("Consumo adicional promedio por flag alimen_pla:")
print(cons_alim)

if {"S","N"}.issubset(cons_alim.index):
    veredicto_h8 = "CONFIRMADA" if cons_alim["S"] > cons_alim["N"] else "REFUTADA"
    linea = f"H8: cons_adicional(S)=${cons_alim['S']:,.0f} vs cons_adicional(N)=${cons_alim['N']:,.0f} -> {veredicto_h8}"
else:
    veredicto_h8 = "INDETERMINADA"
    linea = "H8: no se pudo evaluar (faltan flags alimen_pla)."
print(linea)
report_lines.append("## H8\n" + linea + "\n")

## Tabla resumen de hipótesis

Consolidamos los veredictos en un dataframe.

In [ ]:
resumen_h = pd.DataFrame({
    "hipotesis": ["H1", "H2", "H3", "H4", "H5", "H6", "H7", "H8"],
    "veredicto": [veredicto_h1, veredicto_h2, veredicto_h3, veredicto_h4,
                  veredicto_h5, veredicto_h6, veredicto_h7, veredicto_h8],
})
resumen_h

In [ ]:
resumen_h.to_csv("../reports/hipotesis_resumen.csv", index=False)
with open(REPORT_PATH, "w", encoding="utf-8") as f:
    f.write("\n".join(report_lines))
print(f"Reporte guardado en {REPORT_PATH}")

# 3. Análisis comparativo pre-pandemia / pandemia / post-pandemia

In [ ]:
comp = df.groupby("periodo_covid").agg(
    ingresos = ("ingreso_total", "sum"),
    reservas = ("numvoucher", "nunique"),
    ticket_medio = ("ingreso_total", "mean"),
    duracion_media = ("duracion_estancia", "mean"),
).round(2)
comp

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
comp["ingresos"].plot(kind="bar", ax=axes[0], color=["steelblue","coral","seagreen"])
axes[0].set_title("Ingresos totales por periodo")
axes[0].tick_params(axis="x", rotation=0)

comp["reservas"].plot(kind="bar", ax=axes[1], color=["steelblue","coral","seagreen"])
axes[1].set_title("Cantidad de reservas únicas por periodo")
axes[1].tick_params(axis="x", rotation=0)
plt.tight_layout()
plt.savefig(FIG_DIR / "18_comparativo_periodo.png", dpi=120)
plt.show()

# 4. Resumen ejecutivo

Síntesis de los principales hallazgos del análisis descriptivo. Esta es la salida que se conecta directamente con el informe final y el dashboard.

In [ ]:
print("==============================================")
print("        RESUMEN EJECUTIVO - HOTEL DANN MONASTERIO")
print("==============================================\n")
print(f"Ingresos totales del periodo:     ${kpi01_total:,.0f} COP")
print(f"ADR (tarifa media diaria):        ${kpi02_adr:,.0f} COP/noche")
print(f"Tasa de ocupación:                {kpi03_ocupacion:.2f}%")
print(f"RevPAR:                           ${kpi04_revpar:,.0f} COP")
print(f"Duración promedio de estancia:    {kpi05:.2f} noches")
print(f"Lead time promedio:               {kpi06:.2f} días")
print(f"Ticket promedio por reserva:      ${kpi09:,.0f} COP")
print(f"% consumos adicionales:           {kpi10:.2f}%")
print("\nHipótesis confirmadas:")
print(resumen_h[resumen_h['veredicto']=='CONFIRMADA'])
print("\nHipótesis refutadas:")
print(resumen_h[resumen_h['veredicto']=='REFUTADA'])

# Conclusiones del notebook 04

1. El análisis descriptivo entregó **10 KPIs cuantificados** que ya pueden alimentar los dashboards de Power BI / Tableau.
2. De las **8 hipótesis** formuladas en el Capítulo 5 del anteproyecto, los veredictos quedan registrados en `reports/hipotesis_resumen.csv` y `reports/evaluacion_hipotesis.md`.
3. El **comparativo por periodo COVID** muestra claramente el impacto de la pandemia y la magnitud de la recuperación.
4. La combinación de KPIs + hipótesis validadas constituye la evidencia que se presenta al hotel para apoyar decisiones de:
   - Renegociación de comisiones con canales OTAs.
   - Diseño de campañas dirigidas al rango etario y segmento dominantes.
   - Ajuste de tarifas en temporada alta vs. baja.
   - Estrategias de upselling de consumos adicionales en planes con alimentación.

**Próximos pasos del proyecto**: construcción de los tres dashboards (visión ejecutiva, análisis comercial, análisis operativo) usando los archivos `kpis_resumen.csv` y `hipotesis_resumen.csv` como fuente, y redacción del informe final.